# Phase 2 — QLoRA training run (D043, Amendments 1–3) + DEV sanity check
One notebook for all runs, with **nothing to choose**: the append-only attempt plan picks the next pending run (Amendment 2 item 4) — seed slot 11, then 22, then 33; within a slot the arms EC, then ER, then EE; a slot is fully settled (all three arms valid, or replaced by a reserve that is itself settled) before the next slot starts; reserves come from the queue 44, then 55. Just run all, once per run. Each run: all 160 TRAIN families, 320 examples, **exactly 60 accumulation windows with 60 successful optimizer steps** of 16 examples (8 families) over 3 passes, AdamW lr 1e-4 with 6 warm-up updates then cosine decay to 1e-5 (lr set before each step, gradients unscaled before clipping), GradScaler init 65,536 / growth 2.0 / backoff 0.5 / interval 2,000, the D021 recipe otherwise (NF4 + FP16, LoRA r16/α32/dropout 0.05 on all 7 projections, fp32 adapter). The exposure order is the frozen `phase2/EXPOSURE_ORDER.json`; it and the LoRA initialisation are identical across arms within a seed. About 25 minutes of training plus about 15 minutes of DEV evaluation on a T4.
**Before any attempt opens, the trainer refuses** (nothing is logged) unless: the complete unchanged-model DEV run from `Phase2_Baseline_DEV` is on Drive under the same implementation manifest (Amendment 2 item 6); the configuration and implementation equal those of every earlier attempt (item 7); and the stack is exactly the §2 stack, torch 2.11.0+cu130 included, on a T4 (item 3; there is no override). Inside the attempt it re-verifies `phase2/IMPLEMENTATION_MANIFEST.md` (SHA-256 `18ab0cb5f7ca592e36ecb434b8cd6c7d6e0194d4c412e49912d9bd5e3c813f16`) against its own code, and the TRAIN bytes and the exposure order against their frozen hashes. After update 60 the adapter is saved to Drive with its SHA-256s; then it is evaluated **on DEV as a sanity report only** (never used for selection). Frozen protocol (PREREGISTRATION §4): prompt `test1-english-final-v2`, greedy non-thinking decoding, NF4 + FP16, SDPA math, output cap 1,024 tokens, scorer v1.0.0, engine seed 11. **Do not change any setting.** No FINAL_TEST data is in this notebook.
**Failures (Amendment 1 items 10–11, Amendment 2 item 2).** Every attempt is recorded in the append-only `MyDrive/Hinglish_Phase2/train/ATTEMPT_LOG.jsonl` (hash-chained, never overwritten) and the plan is printed before training.
- *Infrastructure* — only: runtime disconnect or kill, CUDA out-of-memory, a Drive/filesystem `OSError`, a network/Hub download error, a failed checkpoint write. Re-run all: one fresh attempt from scratch with the same seed. After a second infrastructure failure of any arm the whole seed triplet is replaced by the next unused reserve (44, then 55); the plan switches seed by itself. Reserves get the same two-attempt limit and are never reused.
- *Numerical* (GradScaler skip, non-finite loss, gradient or saved parameter) or *protocol* (any hash, identity, label, data or step-count assertion, and **any other error**, e.g. a `TypeError`): the run is invalid and **the study halts** — no retry, no reserve. The notebook then refuses all further training. Send the ZIP.
- A run continued with `RESUME_AFTER_DISCONNECT = True` is never valid for the FinalTest gate.
**One writer per training root (Amendment 3 item 2).** The trainer holds `train/WRITER.lock` (session ID and a heartbeat every 30 s) while it runs and writes its STARTED report before loading the model. A second session is refused, and an attempt that was started but has no recorded outcome is never treated as abandoned automatically: the notebook stops and shows the lock holder and its last heartbeat. If a runtime was killed, first make sure that **no other Colab session** is running this notebook, then set `RECOVER_STALE_WRITER = True` and re-run all: the recovery is logged (`train/WRITER_RECOVERIES.jsonl`), the open attempt is closed as an infrastructure failure (the registered retry/reserve rules then apply unchanged) and the lock is released. Set it back to False afterwards.
**Stale lock (Amendment 2 item 5).** If a runtime was killed during an evaluation, re-running may stop with "Run lock exists". First make sure no other Colab session is running this notebook, then set `RECOVER_STALE_LOCK = True` and re-run all: the lock is removed, a timestamped record is written next to the run, and generation resumes from the stored journal (nothing is regenerated). Set it back to False afterwards.
**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. A ZIP (adapter hashes, training log, attempt log, DEV results; weights stay on Drive) downloads at the end; send it unchanged.


In [ ]:
import os, sys, json, base64, io, zipfile, hashlib, subprocess, time
from pathlib import Path
USE_DRIVE = True  # adapters, checkpoints, the attempt log and the unchanged-DEV baseline live on Drive; keep this on
RESUME_AFTER_DISCONNECT = False  # engineering only: a resumed run is NEVER valid for the FinalTest gate
RECOVER_STALE_WRITER = False  # Amendment 3 item 2: set True ONLY after confirming that no other Colab session is running the trainer (a killed runtime left WRITER.lock or an open attempt); the recovery is logged, the open attempt is closed as an infrastructure failure and the lock is released
RECOVER_STALE_LOCK = False  # Amendment 2 item 5: set True ONLY after confirming that no other Colab session is running this evaluation (a killed runtime left RUNNING.lock); the recovery is logged with a timestamp and stored outputs are never regenerated
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['PYTHONHASHSEED'] = '0'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'  # download bars would become thousands of streamed lines
PACKAGE = Path('/content/hinglish_phase2') if Path('/content').exists() else Path.cwd() / 'hinglish_phase2'
PACKAGE.mkdir(parents=True, exist_ok=True)
blob = base64.b64decode('...')
assert hashlib.sha256(blob).hexdigest() == '99d0dc7c5e6c40056145c34a93f407191c4f5e6625a77ec891063d269d389c6f'
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(PACKAGE)
INPUTS = PACKAGE / 'inputs'
# Implementation manifest (PREREGISTRATION Amendment 1 item 14): refuse to run if the embedded code differs from it.
MANIFEST_TEXT = base64.b64decode('...').decode('utf-8')
MANIFEST_SHA256 = '18ab0cb5f7ca592e36ecb434b8cd6c7d6e0194d4c412e49912d9bd5e3c813f16'
assert hashlib.sha256(MANIFEST_TEXT.encode('utf-8')).hexdigest() == MANIFEST_SHA256, 'Embedded IMPLEMENTATION_MANIFEST.md was changed: refusing to run'
MANIFEST = json.loads(MANIFEST_TEXT.split('```json\n', 1)[1].split('\n```', 1)[0])
assert MANIFEST['preregistration']['sha256'] == 'a22c81fb33b4d3339770a2b28c51d8d3706af2b5dd038d46aa89fbb56139714d', 'Manifest is for another PREREGISTRATION.md: refusing to run'
assert MANIFEST['notebook_payloads']['Phase2_Train.ipynb'] == hashlib.sha256(blob).hexdigest(), 'Notebook payload differs from the implementation manifest: refusing to run'
for member, source in {'hinglish_pilot/__init__.py': 'test1_pilot/hinglish_pilot/__init__.py', 'hinglish_pilot/data.py': 'test1_pilot/hinglish_pilot/data.py', 'hinglish_pilot/engine.py': 'test1_pilot/hinglish_pilot/engine.py', 'hinglish_pilot/hf_backend.py': 'test1_pilot/hinglish_pilot/hf_backend.py', 'hinglish_pilot/report.py': 'test1_pilot/hinglish_pilot/report.py', 'hinglish_pilot/scorer_v1.py': 'test1_pilot/hinglish_pilot/scorer_v1.py', 'secondary_analysis/__init__.py': 'secondary_analysis/__init__.py', 'secondary_analysis/cmi.py': 'secondary_analysis/cmi.py', 'secondary_analysis/numeric_endpoint.py': 'secondary_analysis/numeric_endpoint.py', 'secondary_analysis/report_secondary.py': 'secondary_analysis/report_secondary.py', 't4_smoke/train_smoke.py': 't4_smoke/train_smoke.py', 't4_smoke/labels.py': 't4_smoke/labels.py', 't4_smoke/protocol.py': 't4_smoke/protocol.py', 't4_smoke/train_phase2.py': 't4_smoke/train_phase2.py', 't4_smoke/phase2_eval.py': 't4_smoke/phase2_eval.py', 'inputs/dev.v2.json': 'phase2/dev.v2.json', 'inputs/EXPOSURE_ORDER.json': 'phase2/EXPOSURE_ORDER.json', 'inputs/train_A.v2.json': 'phase2/train_A.v2.json', 'inputs/train_B.v2.json': 'phase2/train_B.v2.json'}.items():
    assert hashlib.sha256((PACKAGE / member).read_bytes()).hexdigest() == MANIFEST['files'][source], f'{member} differs from the implementation manifest: refusing to run'
(INPUTS / 'IMPLEMENTATION_MANIFEST.md').write_text(MANIFEST_TEXT, encoding='utf-8')
print('Implementation manifest verified:', MANIFEST_SHA256)
if USE_DRIVE and Path('/content').exists():  # Drive only inside Colab
    from google.colab import drive
    drive.mount('/content/drive')
    STORE = Path('/content/drive/MyDrive/Hinglish_Phase2')
else:
    STORE = PACKAGE / 'runs'
STORE.mkdir(parents=True, exist_ok=True)
try:  # Qwen3-4B is ungated; a token only avoids anonymous rate limits.
    from google.colab import userdata
    os.environ['HF_TOKEN'] = (userdata.get('HF_TOKEN') or '').strip() or os.environ.get('HF_TOKEN', '')
except Exception as error:
    print('No HF_TOKEN Colab secret (' + type(error).__name__ + '); downloading anonymously.')
if not os.environ.get('HF_TOKEN'):
    os.environ.pop('HF_TOKEN', None)

def step(log_dir, script, *args):
    """Run one phase in a fresh Python process, stream its output and log the attempt."""
    log_dir = Path(log_dir)
    log_dir.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, '-u', str(PACKAGE / 't4_smoke' / script), *map(str, args)]
    started, returncode = time.time(), None
    try:
        with open(log_dir / 'log.txt', 'a') as log, subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                                     text=True, errors='replace') as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
            except BaseException:  # interrupt: never leave a phase running on the GPU behind the notebook
                process.kill()
                process.wait()
                log.write('[notebook] interrupted; phase process killed\n')
                raise
        returncode = process.returncode
    finally:
        with open(log_dir / 'attempts.jsonl', 'a') as attempts:
            attempts.write(json.dumps({'script': script, 'args': list(map(str, args)), 'returncode': returncode,
                                       'started': started, 'seconds': time.time() - started}) + '\n')
    return returncode

def zip_results(folder, archive, prefix):
    """All small files under ``folder``; weights, optimizer state and per-attempt journals stay out (hashes listed)."""
    files, large = [], {}
    for path in sorted(Path(folder).rglob('*')):
        if not path.is_file() or 'journal' in path.relative_to(folder).parts or path.name.endswith('.lock'):
            continue
        if path.suffix in ('.pt', '.safetensors'):
            large[str(path.relative_to(folder))] = hashlib.sha256(path.read_bytes()).hexdigest()
        else:
            files.append(path)
    sums = ''.join(f'{hashlib.sha256(p.read_bytes()).hexdigest()}  {p.relative_to(folder)}\n' for p in files)
    sums += ''.join(f'{digest}  {name}  (not included)\n' for name, digest in large.items())
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        for path in files:
            z.write(path, str(Path(prefix) / path.relative_to(folder)))
        z.writestr(str(Path(prefix) / 'SHA256SUMS.txt'), sums)
    print(sums)
    print('Return:', archive, 'SHA-256', hashlib.sha256(Path(archive).read_bytes()).hexdigest())
    try:
        from google.colab import files as colab_files
        colab_files.download(str(archive))
    except ImportError:
        print('Local: ZIP saved on disk.')


In [ ]:
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:  # CPU runtime: no nvidia-smi at all
    gpu = ''
print('GPU:', gpu or 'none')
assert 'T4' in gpu, 'Select Runtime → Change runtime type → T4 GPU'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PACKAGE / 'requirements-phase2.txt')], check=True)
# A fresh interpreter sees the installed versions; this kernel never imports torch.
STACK = json.loads(subprocess.run([sys.executable, '-c', 'import sys; sys.path.insert(0, sys.argv[1]); import train_smoke as t, json; '
                                   'v = t.versions(); print(json.dumps({"versions": v, "mismatches": t.stack_mismatches(v)}))',
                                   str(PACKAGE / 't4_smoke')], capture_output=True, text=True, check=True).stdout)
print(json.dumps(STACK['versions'], indent=1))
STACK_MISMATCHES = STACK['mismatches']
print('D021 stack mismatches:', STACK_MISMATCHES or 'none')
# Amendment 2 item 3: only the exact §2 stack on a T4 can produce a study run; there is no override.
assert not STACK_MISMATCHES, 'D021 stack mismatch: no study run until a dated amendment (Amendment 2 item 3): ' + '; '.join(STACK_MISMATCHES)
# Tokenizer, chat template and config files at the pinned revision must match the implementation manifest.
HUB_CHECK = """
import hashlib, json, os, sys
from huggingface_hub import hf_hub_download
model = json.loads(sys.argv[1])
for name, expected in sorted(model['hub_files'].items()):
    path = hf_hub_download(model['id'], name, revision=model['revision'], token=os.environ.get('HF_TOKEN') or None)
    actual = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    assert actual == expected, f'{name} at the pinned revision has SHA-256 {actual}, manifest {expected}: refusing to run'
print('Pinned Hub files match the implementation manifest:', ', '.join(sorted(model['hub_files'])))
"""
print(subprocess.run([sys.executable, '-c', HUB_CHECK, json.dumps(MANIFEST['model'])], capture_output=True, text=True, check=True).stdout)


In [ ]:
sys.path.insert(0, str(PACKAGE / 't4_smoke'))
import train_phase2 as tp
TRAIN_ROOT = STORE / 'train'
BASELINE_DIR = STORE / 'baseline_dev'  # Amendment 2 item 6: written by Phase2_Baseline_DEV
LOG = TRAIN_ROOT / tp.ATTEMPT_LOG
if RECOVER_STALE_WRITER:  # Amendment 3 item 2: only after the operator confirmed that no other session is running
    print('RECOVER_STALE_WRITER = True: the operator confirmed that no other Colab session is running the trainer.')
    tp.recover_stale_writer(TRAIN_ROOT, confirmed_no_other_session=True)
print(tp.describe_writer(TRAIN_ROOT))
assert tp.read_writer_lock(TRAIN_ROOT) is None, tp.writer_held_message(TRAIN_ROOT)
PLAN = tp.plan(tp.read_log(LOG))
print(tp.describe_plan(PLAN))
ACTION = tp.next_run(PLAN)  # an open attempt is refused, never closed automatically (Amendment 3 item 2)  # Amendment 2 item 4: the plan chooses ARM and SEED; refuses after a numerical/protocol failure
assert not STACK_MISMATCHES, 'D021 stack mismatch (no attempt was opened): ' + '; '.join(STACK_MISMATCHES)
ATTEMPT_DIR, OUTCOME = None, None
if ACTION['kind'] == 'complete':
    print(f"Every seed slot is settled (S = {ACTION['S']}): nothing left to train. Next: the adapter freeze (ADAPTERS_FROZEN.md).")
else:
    ARM, SEED, SLOT, ATTEMPT, RESUME = ACTION['arm'], ACTION['seed'], ACTION['slot'], ACTION['attempt'], False
    target = tp.resume_target(PLAN, ACTION, TRAIN_ROOT) if RESUME_AFTER_DISCONNECT else None
    if target is not None:
        ATTEMPT, RESUME = target, True
        print('RESUMING attempt', ATTEMPT, 'for engineering use only: a resumed run is NEVER valid for the FinalTest gate')
    else:
        print(f"Next run from the plan: {ARM} seed {SEED} (slot {SLOT}{', reserve' if ACTION['reserve'] else ''}): attempt {ATTEMPT}"
              + (' — from-scratch retry after an infrastructure failure' if ACTION['retry'] else ''))
    ATTEMPT_DIR = tp.attempt_dir(TRAIN_ROOT, ARM, SEED, ATTEMPT)
    assert RESUME or not (ATTEMPT_DIR / 'train_report.json').exists(), f'{ATTEMPT_DIR} already holds an attempt: attempts are never overwritten'
    args = ['train', '--train-root', TRAIN_ROOT, '--inputs', INPUTS, '--baseline-dir', BASELINE_DIR,
            '--arm', ARM, '--seed', SEED, '--slot', SLOT, '--attempt', ATTEMPT]  # the trainer re-derives and asserts these
    returncode = step(ATTEMPT_DIR, 'train_phase2.py', *args, *(['--resume'] if RESUME else []))
    ENTRIES = tp.read_log(LOG)
    PLAN = tp.plan(ENTRIES)
    print('\n' + tp.describe_plan(PLAN))
    same = lambda e: (e['arm'], e['seed'], e['attempt']) == (ARM, SEED, ATTEMPT)
    opened = [e for e in ENTRIES if e['event'] == ('resume_start' if RESUME else 'start') and same(e)]
    closed = [e for e in ENTRIES if e['event'] == ('resume_finish' if RESUME else 'finish') and same(e)]
    assert opened, f'The trainer refused before opening an attempt (exit {returncode}; see its message above). Nothing was logged.'
    assert closed, 'The attempt has no recorded outcome (runtime killed?). Make sure no other session is running, then set RECOVER_STALE_WRITER = True and re-run all.'
    OUTCOME, DETAIL = closed[-1]['outcome'], closed[-1].get('detail')
    if OUTCOME in ('numerical', 'protocol'):
        print('\n' + '!' * 100 + f'\n{OUTCOME.upper()} FAILURE — {DETAIL}\n' + tp.HALT_MESSAGE + '\n' + '!' * 100)
    assert OUTCOME == 'valid', {
        'infrastructure': f'Infrastructure failure ({DETAIL}). Run the Download cell, send the ZIP, then re-run all: the plan above decides the retry or the reserve seed.',
        'resumed_invalid': 'Resumed run finished but is never valid. Set RESUME_AFTER_DISCONNECT = False and re-run all for a fresh attempt if one is left.',
    }.get(OUTCOME, f'{OUTCOME} failure: the study halts. Run the Download cell and send the ZIP.')
    FINAL = ATTEMPT_DIR / 'final_adapter.json'
    final = json.loads(FINAL.read_text())
    print(json.dumps({k: final[k] for k in ('arm', 'seed', 'slot', 'attempt', 'steps', 'adapter_model_sha256', 'adapter_config_sha256',
                                            'initial_adapter_sha256', 'order_sha256', 'resumed', 'valid_for_gate',
                                            'baseline_dev_results_zip_sha256')}, indent=1))


In [ ]:
# DEV sanity report only (PREREGISTRATION §5): never selects checkpoints, arms or reruns.
if OUTCOME == 'valid':
    DEV_RUN = ATTEMPT_DIR / 'dev_eval'
    step(ATTEMPT_DIR, 'phase2_eval.py', 'evaluate', '--split', 'dev', '--inputs', INPUTS, '--run-dir', DEV_RUN,
         '--adapter-dir', ATTEMPT_DIR / 'final_adapter', '--adapter-record', FINAL,
         '--implementation-manifest', INPUTS / 'IMPLEMENTATION_MANIFEST.md', *(['--recover-stale-lock'] if RECOVER_STALE_LOCK else []))
else:
    print('Nothing was trained in this session: no DEV sanity run.')


## Download
Run this cell even after a failure or a refusal: the ZIP carries the error, the logs and a snapshot of the attempt log.


In [ ]:
import shutil
if globals().get('ATTEMPT_DIR') is not None and ATTEMPT_DIR.exists():
    if LOG.exists():
        shutil.copyfile(LOG, ATTEMPT_DIR / 'ATTEMPT_LOG.snapshot.jsonl')
    zip_results(ATTEMPT_DIR, PACKAGE / f'Phase2_Train_{ARM}-s{SEED}_{ATTEMPT_DIR.name}.zip', f'{ARM}-s{SEED}/{ATTEMPT_DIR.name}')
else:  # refused before an attempt was opened (e.g. study halted, no baseline): send the whole training log tree
    zip_results(STORE / 'train', PACKAGE / 'Phase2_Train_logs.zip', 'train')
